# Verify the scientific runtime

Probe the configured runtime in a separate process. Checks all 119 package versions, pip consistency and CPU arithmetic. No Unity launch, model loading or training. Missing configuration is saved as BLOCKED; a configured runtime mismatch raises an error.


In [ ]:
from pathlib import Path
import os
_candidate = Path(os.environ.get('TRACE_LAB_ROOT', Path.cwd())).expanduser().resolve()
_candidates = [_candidate] if os.environ.get('TRACE_LAB_ROOT') else [_candidate, *_candidate.parents]
ROOT = next((p for p in _candidates if (p / '.trace-lab-root').is_file() and (p / 'configs').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open inside trace-lab or set TRACE_LAB_ROOT to the clone')
# Run definition cells in this fresh kernel; this notebook has no execution side effects.
get_ipython().run_line_magic('run', '"' + str(ROOT / 'notebooks/library/configuration.ipynb') + '"')
ROOT = workspace_root(ROOT)


## Inspect the configured interpreter


In [ ]:
assets = load_assets(ROOT)
print(json.dumps(asset_status(assets), indent=2))


## Verify and save a uniquely named local record


In [ ]:
state = next(row for row in asset_status(assets) if row['asset'] == 'runtime_python')
if state['status'] != 'AVAILABLE':
    result = {'status': 'BLOCKED', 'reason': state, 'scope': 'Configure runtime_python; no runtime check executed.'}
else:
    probe = runtime_probe(require_asset(assets, 'runtime_python'))
    result = compare_runtime(ROOT, probe)
    result['probe'] = probe
record = save_record(ROOT, 'runtime_verification', result)
print(json.dumps({key: value for key, value in result.items() if key != 'probe'}, indent=2))
print('Local evidence:', record)
if result['status'] == 'FAIL':
    raise AssertionError('Runtime differs from the preserved environment; see saved record')
